# Alkahest 4B RP Qwen Text Export

Builds the definitive text-only q4 WebGPU ONNX package from `thomasjvu/alkahest-4b-heretic-rp-merged` as `thomasjvu/alkahest-4b-heretic-q4-onnx-rp-text`.

In [ ]:
import os, platform, shutil, subprocess, sys, time
from pathlib import Path

SOURCE_REPO_ID = os.environ.get('SOURCE_REPO_ID', 'thomasjvu/alkahest-4b-heretic-rp-merged')
TEMPLATE_MODEL_ID = os.environ.get('TEMPLATE_MODEL_ID', 'onnx-community/Qwen3.5-4B-ONNX-OPT')
BASE_MODEL_ID = os.environ.get('BASE_MODEL_ID', 'Qwen/Qwen3.5-4B')
TARGET_REPO_ID = os.environ.get('TARGET_REPO_ID', 'thomasjvu/alkahest-4b-heretic-q4-onnx-rp-text')
WORK_DIR = Path(os.environ.get('TEXT_EXPORT_WORK_DIR', '/kaggle/working/alkahest-4b-rp-qwen-text-export'))
source_candidates = [
    os.environ.get('SOURCE_REPO_ID', ''),
    '/kaggle/input/alkahest-4b-two-stage-sft-t4/alkahest-4b-two-stage-sft/stage-ab-merged',
    '/kaggle/input/alkahest-4b-two-stage-sft/alkahest-4b-two-stage-sft/stage-ab-merged',
]
source_candidates.extend(str(path.parent) for path in Path('/kaggle/input').glob('**/stage-ab-merged/config.json'))
print('source_candidates=', source_candidates)
for candidate in source_candidates:
    if candidate and Path(candidate).exists():
        SOURCE_REPO_ID = candidate
        break

print('python_platform=', platform.platform())
print('working_disk_free_gb=', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))
print('resolved_source_repo_id=', SOURCE_REPO_ID)
print('target_repo_id=', TARGET_REPO_ID)

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
secret_token = ''
for attempt in range(5):
    try:
        from kaggle_secrets import UserSecretsClient
        secret_token = UserSecretsClient().get_secret('HF_TOKEN')
        break
    except Exception as exc:
        print('hf_secret_attempt_failed=', attempt + 1, type(exc).__name__)
        time.sleep(3)
if secret_token and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = secret_token
if secret_token and not os.environ.get('HUGGING_FACE_HUB_TOKEN'):
    os.environ['HUGGING_FACE_HUB_TOKEN'] = secret_token
print('hf_secret_loaded=', bool(secret_token))

packages = [
    'huggingface_hub[cli]>=1.5.0',
    'hf_transfer>=0.1.9',
    'safetensors>=0.7.0',
    'onnx>=1.19.0',
    'onnx-ir>=0.1.12',
    'onnxruntime>=1.23.0',
    'pyyaml>=6.0.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
if token:
    try:
        from huggingface_hub import HfApi
        print('hf_whoami=', HfApi(token=token).whoami().get('name', 'unknown'))
    except Exception as exc:
        print('hf_token_invalid=', type(exc).__name__)
        os.environ.pop('HF_TOKEN', None)
        os.environ.pop('HUGGING_FACE_HUB_TOKEN', None)
print('hf_token_present=', bool(os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')))


In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = os.environ.get('HERETIC_TO_ONNX_REPO', 'https://github.com/alkahest-ai/heretic-to-onnx.git')
REPO_REF = os.environ.get('HERETIC_TO_ONNX_REF', 'codex/kaggle-heretic-2b-run')
REPO_DIR = Path('/kaggle/working/heretic-to-onnx')
if REPO_DIR.exists():
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--branch', REPO_REF, '--depth', '1', REPO_URL, str(REPO_DIR)])
print('repo=', REPO_DIR)
print('head=', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())

cmd = [
    sys.executable,
    str(REPO_DIR / 'scripts/kaggle_alkahest_qwen_text_export.py'),
    '--source-repo-id', SOURCE_REPO_ID,
    '--template-model-id', TEMPLATE_MODEL_ID,
    '--base-model-id', BASE_MODEL_ID,
    '--target-repo-id', TARGET_REPO_ID,
    '--work-dir', str(WORK_DIR),
]
if os.environ.get('TEXT_EXPORT_PUBLIC') == '1':
    cmd.append('--no-private')
if os.environ.get('TEXT_EXPORT_NO_UPLOAD') == '1':
    cmd.append('--no-upload')
elif not (os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')):
    cmd.append('--no-upload')
if os.environ.get('TEXT_EXPORT_NO_RUNTIME_SMOKE') == '1':
    cmd.append('--no-runtime-smoke')
print('running=', ' '.join(cmd))
subprocess.check_call(cmd)

import json
report_path = WORK_DIR / 'package' / TARGET_REPO_ID.replace('/', '__') / 'text-export-report.json'
print('report_path=', report_path)
print(json.dumps(json.loads(report_path.read_text()), indent=2)[:4000])
